In [1]:
import re 
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [2]:
from bs4 import BeautifulSoup

In [3]:
with open("apple_10k_2023.htm", 'r', encoding='utf-8' ,errors='ignore') as f:
    html  = f.read()

In [4]:
soup = BeautifulSoup(html,'html.parser')
for tag in soup(['script','style','meta','link']):
    tag.decompose()

In [5]:
print("HTML loaded and cleaned")
print(f"Total text length: {len(soup.get_text()):,} characters")

HTML loaded and cleaned
Total text length: 213,113 characters


#  Extracting content

In [6]:
def extract_sections_semantic(soup) -> list[dict]:
    """
    semantic means finding natural documenting boundaries instead of arbitrary
    character counts  splits financial documments have clear section header 
    """

    sections = []
    section_num= 0

    all_elements = soup.find_all(['div','p','section'])

    current_title = "Document Header"
    current_text = []
    current_length = 0

    for elem in all_elements:
        if elem.find(['div','section']):
            continue

        text = elem.get_text(strip=True)

        if len(text) < 50:
            continue

        is_header = (
            elem.name in ['h1', 'h2', 'h3', 'h4'] or
            (len(text) < 100 and elem.name == 'p' and 
             any(keyword in text.upper() for keyword in [
                 'CONSOLIDATED', 'STATEMENTS', 'NOTES', 
                 'PART', 'ITEM', 'REPORT', 'BALANCE SHEET',
                 'OPERATIONS', 'CASH FLOW'
             ]))
        )

        if is_header and current_text:
            # Save current section
            if current_length > 200:
                section_num += 1
                sections.append({
                    "section_num": section_num,
                    "section_title": current_title,
                    "text": " ".join(current_text),
                    "char_count": current_length
                })
            current_title = text[:100]
            current_text = []
            current_length = 0
        else:
            current_text.append(text)
            current_length += len(text)

            if current_length > 1500:
                section_num += 1
                sections.append({
                    "section_num": section_num,
                    "section_title": current_title,
                    "text": " ".join(current_text),
                    "char_count": current_length
                })
                current_text = []
                current_length = 0
    
    return sections

In [7]:
sections = extract_sections_semantic(soup)

In [8]:
print(f"Sections extracted: {len(sections)}")
print(f"Total characters: {sum(s['char_count'] for s in sections):,}")
print(f"\nSection titles found:")
for s in sections[:10]:
    print(f"  Section {s['section_num']}: '{s['section_title'][:60]}' — {s['char_count']} chars")

Sections extracted: 98
Total characters: 196,102

Section titles found:
  Section 1: 'Document Header' — 15207 chars
  Section 2: 'Document Header' — 1646 chars
  Section 3: 'Document Header' — 1909 chars
  Section 4: 'Document Header' — 2639 chars
  Section 5: 'Document Header' — 1537 chars
  Section 6: 'Document Header' — 1503 chars
  Section 7: 'Document Header' — 1517 chars
  Section 8: 'Document Header' — 2616 chars
  Section 9: 'Document Header' — 1920 chars
  Section 10: 'Document Header' — 1766 chars


In [9]:
headings = soup.find_all(['h1', 'h2', 'h3', 'h4'])
print(f"Total heading tags found: {len(headings)}")
for h in headings[:10]:
    print(f"  <{h.name}>: '{h.get_text(strip=True)[:80]}'")

print("\nShort paragraphs that might be headers:")
short_paras = [p for p in soup.find_all('p') 
               if 10 < len(p.get_text(strip=True)) < 100]
for p in short_paras[:15]:
    print(f"  '{p.get_text(strip=True)[:80]}'")

Total heading tags found: 0

Short paragraphs that might be headers:


In [16]:
index_soup = BeautifulSoup(response.text, 'html.parser')

In [21]:
def extract_sections_from_xbrl_html(soup) -> list[dict]:
    sections = []
    section_num = 0
    current_title = "Preamble"
    current_text = []
    current_length = 0
    
    all_elements = soup.find_all(['span', 'div', 'p', 'ix:nonnumeric', 'ix:nonfraction'])
    
    for elem in all_elements:
        if elem.find(['span', 'div']):
            continue
            
        text = elem.get_text(strip=True)
        if len(text) < 5:
            continue
        
        style = elem.get('style', '')
        is_header = (
            'font-weight:700' in style and
            len(text) < 150 and
            any(text.upper().startswith(k) for k in [
                'ITEM', 'PART', 'CONSOLIDATED', 'NOTE',
                'MANAGEMENT', 'REPORT', 'SELECTED'
            ])
        )
        
        if is_header and current_text and current_length > 200:
            section_num += 1
            sections.append({
                "section_num": section_num,
                "section_title": current_title,
                "text": " ".join(current_text),
                "char_count": current_length
            })
            current_title = text[:100]
            current_text = []
            current_length = 0
        else:
            current_text.append(text)
            current_length += len(text)
            
            if current_length > 1500:
                section_num += 1
                sections.append({
                    "section_num": section_num,
                    "section_title": current_title,
                    "text": " ".join(current_text),
                    "char_count": current_length
                })
                current_text = []
                current_length = 0
    
    return sections

sections = extract_sections_from_xbrl_html(soup)
print(f"Sections extracted: {len(sections)}")
print(f"\nFirst 10 section titles:")
for s in sections[:10]:
    print(f"  Section {s['section_num']}: '{s['section_title'][:70]}' — {s['char_count']} chars")

Sections extracted: 133

First 10 section titles:
  Section 1: 'Preamble' — 15207 chars
  Section 2: 'Preamble' — 1502 chars
  Section 3: 'Preamble' — 1503 chars
  Section 4: 'Preamble' — 1743 chars
  Section 5: 'Preamble' — 1541 chars
  Section 6: 'Preamble' — 1012 chars
  Section 7: 'Part I' — 729 chars
  Section 8: 'Part III' — 319 chars
  Section 9: 'Part IV' — 1613 chars
  Section 10: 'Part IV' — 443 chars


In [22]:
titles = {}
for s in sections:
    title = s['section_title'][:60]
    if title not in titles:
        titles[title] = 0
    titles[title] += 1

print("Unique section titles and count:")
for title, count in sorted(titles.items(), key=lambda x: -x[1]):
    print(f"  {count:3d}x — '{title}'")

Unique section titles and count:
   43x — 'PART I'
    9x — 'Item 6.    [Reserved]'
    8x — 'PART IV'
    7x — 'Note 4 –Financial Instruments'
    6x — 'Preamble'
    6x — 'Report of Independent Registered Public Accounting Firm'
    5x — 'Note 8 –Leases'
    4x — 'Note 2 –Revenue'
    4x — 'Note 7 –Income Taxes'
    3x — 'Note 11 –Share-Based Compensation'
    3x — 'Note 13 –Segment Information and Geographic Data'
    2x — 'Part IV'
    2x — 'Item 3.    Legal Proceedings'
    2x — 'Item 4.    Mine Safety Disclosures'
    2x — 'Item 7A.    Quantitative and Qualitative Disclosures About M'
    2x — 'CONSOLIDATED BALANCE SHEETS'
    2x — 'CONSOLIDATED STATEMENTS OF CASH FLOWS'
    2x — 'Notes to Consolidated Financial Statements'
    2x — 'Note 12 –Commitments, Contingencies and Supply Concentration'
    2x — 'Item 9.    Changes in and Disagreements with Accountants on '
    2x — 'Item 9B.    Other Information'
    1x — 'Part I'
    1x — 'Part III'
    1x — 'Item 1B.    Unresolved Staf

# Citation 

In [29]:
def build_chunks_with_citations(sections: list[dict],
                               company: str,
                               doc_type:str,
                               fiscal_year:str,
                               source_file:str) -> list[dict]:

    # every chunk must carry full provenance so LLM can say exactly where each answer came from
    financial_signals = [
        "revenue", "income", "profit", "loss", "sales", "earnings",
        "expense", "cost", "margin", "cash", "debt", "equity",
        "assets", "liabilities", "operating", "net", "gross", "total",
        "billion", "million", "fiscal", "quarter", "annual", "growth",
        "depreciation", "amortization", "ebitda", "eps", "dividend"
    ]

    chunks = []
    for section in sections:
        # WHY: XBRL metadata contains URLs, real financial text doesn't
        if 'http://' in section["text"] or 'https://' in section["text"]:
            continue
        text_lower = section["text"].lower()
        signals_found = sum(1 for w in financial_signals if w in text_lower)
        quality_score = signals_found/ len(financial_signals)

        if quality_score < 0.2:
            continue

        chunks.append({
            "chunk_id": f"{company.lower().replace(' ', '_').replace('.', '')}_{fiscal_year}_s{section['section_num']:03d}",            "text": section["text"],
            "char_count": section["char_count"],
            "company": company,
            "doc_type": doc_type,
            "fiscal_year": fiscal_year,
            "section_title": section["section_title"],
            "section_num": section["section_num"],
            "source_file": source_file,
            "quality_score": round(quality_score, 3)
        })

    return chunks

In [30]:
chunks = build_chunks_with_citations(
    sections,
    company="Apple Inc.",
    doc_type="10-K",
    fiscal_year="FY2023",
    source_file="aapl-20230930.htm"
)

In [31]:
print(f"Total chunks after quality filter: {len(chunks)}")
print(f"\nSample citations:")
for chunk in chunks[:5]:
    print(f"\n  ID: {chunk['chunk_id']}")
    print(f"  Citation: {chunk['company']} {chunk['doc_type']} {chunk['fiscal_year']}")
    print(f"  Section: {chunk['section_title']}")
    print(f"  Quality: {chunk['quality_score']}")
    print(f"  Preview: {chunk['text'][:100]}")

Total chunks after quality filter: 38

Sample citations:

  ID: apple_inc_FY2023_s013
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.241
  Preview: Markets and Distribution The Company’s customers are primarily in the consumer, small and mid-sized 

  ID: apple_inc_FY2023_s016
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.241
  Preview: The Company regularly files patent, design, copyright and trademark applications to protect innovati

  ID: apple_inc_FY2023_s024
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.207
  Preview: The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely 

  ID: apple_inc_FY2023_s029
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.207
  Preview: The Company’s products and services may be affected from time to time by design and manufacturing de

  ID: apple_inc_FY2023_s042
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.276


In [32]:
# Find the key financial statement chunks
print("Financial statement chunks:")
key_sections = [
    "CONSOLIDATED STATEMENTS OF OPERATIONS",
    "CONSOLIDATED BALANCE SHEETS", 
    "CONSOLIDATED STATEMENTS OF CASH FLOWS",
    "Note 2",
    "Note 13"
]

for chunk in chunks:
    if any(key in chunk['section_title'] for key in key_sections):
        print(f"\n  ID: {chunk['chunk_id']}")
        print(f"  Section: {chunk['section_title']}")
        print(f"  Quality: {chunk['quality_score']}")
        print(f"  Preview: {chunk['text'][:150]}")

Financial statement chunks:

  ID: apple_inc_FY2023_s071
  Section: CONSOLIDATED STATEMENTS OF OPERATIONS
  Quality: 0.379
  Preview: (In millions, except number of shares, which are reflected in thousands, and per-share amounts) Years ended September 30,2023 September 24,2022 Septem

  ID: apple_inc_FY2023_s073
  Section: CONSOLIDATED BALANCE SHEETS
  Quality: 0.31
  Preview: (In millions, except number of shares, which are reflected in thousands, and par value) September 30,2023 September 24,2022 ASSETS: Current assets: Ca

  ID: apple_inc_FY2023_s076
  Section: CONSOLIDATED STATEMENTS OF CASH FLOWS
  Quality: 0.414
  Preview: (In millions) Years ended September 30,2023 September 24,2022 September 25,2021 Cash, cash equivalents and restricted cash, beginning balances 24,977 

  ID: apple_inc_FY2023_s082
  Section: Note 2 –Revenue
  Quality: 0.241
  Preview: For the sale of third-party products where the Company obtains control of the product before transferring it to the customer, th

# Store in the chroma DB

In [34]:
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading embedding model...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1886.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
client = chromadb.PersistentClient(path="./chroma_db/")
try:
    client.delete_collection("financial_doc_v2")
except:
    pass

In [37]:
collection = client.create_collection(
    name="financial_docs_v2",
    metadata={"hnsw:space": "cosine"}
)

In [38]:
texts = [chunk["text"] for chunk in chunks ]

In [39]:
print(f"Embedding {len(chunks)} chunks...")

Embedding 38 chunks...


In [40]:
embeddings = model.encode(texts, batch_size=32,show_progress_bar=True)
print(f"Embedding shape: {embeddings.shape}")

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.15s/it]

Embedding shape: (38, 384)


In [41]:
collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{
        "company": chunk["company"],
        "doc_type": chunk["doc_type"],
        "fiscal_year": chunk["fiscal_year"],
        "section_title": chunk["section_title"],
        "section_num": chunk["section_num"],
        "source_file": chunk["source_file"],
        "quality_score": chunk["quality_score"]
    } for chunk in chunks]
)

print(f"\nStored {collection.count()} chunks with citation metadata")


Stored 38 chunks with citation metadata


# BM25 index over new chunks

In [42]:
tokenized_docs = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_docs)
print(f"BM25 index build : {len(bm25.idf)} unique terms")

BM25 index build : 2276 unique terms


In [50]:
def retreive_with_citations(query: str, 
                             bm25_index, 
                             documents: list,
                             chunks: list,
                             collection,
                             n_results: int = 3,
                             alpha: float = 0.7) -> list[dict]:
    """
    We need citation because it will be easy for the use to verify whether the answer is genuine or not 
    """

    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max > bm25_min:
        bm25_normalized = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
    else:
        bm25_normalized = bm25_scores

    # Dense scores
    dense_results = collection.query(
        query_texts=[query],
        n_results=len(documents)
    )
    dense_distances = np.array(dense_results["distances"][0])
    dense_similarity = 1 - dense_distances
    dense_ids = dense_results["ids"][0]
    all_ids = [chunk["chunk_id"] for chunk in chunks]

    dense_aligned = np.zeros(len(documents))
    for i, doc_id in enumerate(all_ids):
        if doc_id in dense_ids:
            dense_idx = dense_ids.index(doc_id)
            dense_aligned[i] = dense_similarity[dense_idx]

    # Hybrid scores
    hybrid_scores = (alpha * dense_aligned) + ((1 - alpha) * bm25_normalized)
    top_indices = np.argsort(-hybrid_scores)[:n_results]

    results = []
    for idx in top_indices:
        chunk = chunks[idx]
        
        # Build citation string
        # WHY formatted citation: LLM pastes this directly into answer
        citation = (f"{chunk['company']} {chunk['doc_type']} "
                   f"{chunk['fiscal_year']} — {chunk['section_title']}")
        
        results.append({
            "text": chunk["text"],
            "citation": citation,
            "company": chunk["company"],
            "doc_type": chunk["doc_type"],
            "fiscal_year": chunk["fiscal_year"],
            "section_title": chunk["section_title"],
            "hybrid_score": round(float(hybrid_scores[idx]), 4),
            "chunk_id": chunk["chunk_id"]
        })
    
    return results


In [51]:
test_queries = [
    "What is Apple's total revenue for 2023?",
    "What is Apple's net income?",
    "Apple cash flow from operations"]

In [54]:
for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = retreive_with_citations(query, bm25, texts, chunks, collection)
    for r in results[:2]:
        print(f"  [{r['citation']}]")
        print(f"  Score: {r['hybrid_score']} | Preview: {r['text'][:100]}")


Query: 'What is Apple's total revenue for 2023?'
  [Apple Inc. 10-K FY2023 — Item 6.    [Reserved]]
  Score: 0.6631 | Preview: Operating Expenses Operating expenses for 2023, 2022 and 2021 were as follows (dollars in millions):
  [Apple Inc. 10-K FY2023 — Notes to Consolidated Financial Statements]
  Score: 0.6293 | Preview: Revenue The Company records revenue net of taxes collected from customers that are remitted to gover

Query: 'What is Apple's net income?'
  [Apple Inc. 10-K FY2023 — PART I]
  Score: 0.6412 | Preview: The Company regularly files patent, design, copyright and trademark applications to protect innovati
  [Apple Inc. 10-K FY2023 — Item 6.    [Reserved]]
  Score: 0.6045 | Preview: Apple Inc. | 2023 Form 10-K | 21 Products and Services Performance The following table shows net sal

Query: 'Apple cash flow from operations'
  [Apple Inc. 10-K FY2023 — Note 4 –Financial Instruments]
  Score: 0.5382 | Preview: To protect the Company’s foreign currency–denominated term deb

In [53]:
def enrich_chunk_text(chunk: dict) -> str:
    """
    WHY prepend section title: tables have no natural language
    adding the title gives the embedding model context about
    what the numbers mean
    """
    return f"{chunk['section_title']}:\n{chunk['text']}"

# Re-embed with enriched text
enriched_texts = [enrich_chunk_text(chunk) for chunk in chunks]

# Rebuild collection with enriched embeddings
try:
    client.delete_collection("financial_docs_v2")
except:
    pass

collection = client.create_collection(
    name="financial_docs_v2",
    metadata={"hnsw:space": "cosine"}
)

print("Re-embedding with enriched text...")
enriched_embeddings = model.encode(enriched_texts, batch_size=32, show_progress_bar=True)

collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    embeddings=enriched_embeddings.tolist(),
    documents=enriched_texts,
    metadatas=[{
        "company": chunk["company"],
        "doc_type": chunk["doc_type"],
        "fiscal_year": chunk["fiscal_year"],
        "section_title": chunk["section_title"],
        "section_num": chunk["section_num"],
        "source_file": chunk["source_file"],
        "quality_score": chunk["quality_score"]
    } for chunk in chunks]
)

# Rebuild BM25 with enriched text
tokenized_enriched = [text.lower().split() for text in enriched_texts]
bm25 = BM25Okapi(tokenized_enriched)

print(f"Re-indexed {collection.count()} chunks with enriched text")

Re-embedding with enriched text...


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.12s/it]

Re-indexed 38 chunks with enriched text


In [56]:
for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = retreive_with_citations(query, bm25, enriched_texts, chunks, collection)
    for r in results[:2]:
        print(f"  [{r['citation']}]")
        print(f"  Score: {r['hybrid_score']} | Preview: {r['text'][:120]}")


Query: 'What is Apple's total revenue for 2023?'
  [Apple Inc. 10-K FY2023 — Item 6.    [Reserved]]
  Score: 0.6631 | Preview: Operating Expenses Operating expenses for 2023, 2022 and 2021 were as follows (dollars in millions): Change Change Resea
  [Apple Inc. 10-K FY2023 — Notes to Consolidated Financial Statements]
  Score: 0.6293 | Preview: Revenue The Company records revenue net of taxes collected from customers that are remitted to governmental authorities.

Query: 'What is Apple's net income?'
  [Apple Inc. 10-K FY2023 — PART I]
  Score: 0.6412 | Preview: The Company regularly files patent, design, copyright and trademark applications to protect innovations arising from its
  [Apple Inc. 10-K FY2023 — Item 6.    [Reserved]]
  Score: 0.6045 | Preview: Apple Inc. | 2023 Form 10-K | 21 Products and Services Performance The following table shows net sales by category for 2

Query: 'Apple cash flow from operations'
  [Apple Inc. 10-K FY2023 — Note 4 –Financial Instruments]
  Score: 

In [57]:
# Direct lookup — are the financial statement chunks even in the collection?
print("Direct retrieval test — searching for income statement chunk:")
result = collection.get(
    ids=["apple_inc_FY2023_s071"],
    include=["documents", "metadatas"]
)
print(f"Found: {len(result['ids'])} chunk(s)")
if result['ids']:
    print(f"Section: {result['metadatas'][0]['section_title']}")
    print(f"Preview: {result['documents'][0][:200]}")

print("\nDirect query with exact financial terms:")
direct = collection.query(
    query_texts=["total net sales 383285 revenue products services"],
    n_results=3
)
for i in range(len(direct['ids'][0])):
    meta = direct['metadatas'][0][i]
    print(f"  Section: {meta['section_title']} | Distance: {round(direct['distances'][0][i], 4)}")

Direct retrieval test — searching for income statement chunk:
Found: 1 chunk(s)
Section: CONSOLIDATED STATEMENTS OF OPERATIONS
Preview: CONSOLIDATED STATEMENTS OF OPERATIONS:
(In millions, except number of shares, which are reflected in thousands, and per-share amounts) Years ended September 30,2023 September 24,2022 September 25,2021

Direct query with exact financial terms:
  Section: CONSOLIDATED STATEMENTS OF OPERATIONS | Distance: 0.4327
  Section: Note 2 –Revenue | Distance: 0.5063
  Section: Notes to Consolidated Financial Statements | Distance: 0.5086


In [62]:
def expand_financial_query(query: str) -> str:
    """
    query expansion: bridges vocabulary gap between
    natural language questions and numeric financial tables
    """
    expansions = {
        "revenue": "revenue net sales total sales income",
        "profit": "profit net income earnings",
        "cash flow": "cash flow operations operating activities",
        "expenses": "expenses cost of sales operating expenses",
        "assets": "assets current assets total assets",
        "debt": "debt liabilities long-term debt",
        "earnings per share": "earnings per share EPS diluted basic",
        "net income": "net income profit earnings after tax",
        "total revenue": "total revenue net sales total net sales",
    }
    
    expanded = query
    query_lower = query.lower()
    
    for term, expansion in expansions.items():
        if term in query_lower:
            expanded = f"{expanded} {expansion}"
    
    return expanded

# Test query expansion
test_queries_expanded = [
    "What is Apple's total revenue for 2023?",
    "What is Apple's net income?",
    "Apple cash flow from operations"
]

print("Query expansion results:")
for q in test_queries_expanded:
    expanded = expand_financial_query(q)
    print(f"\nOriginal: {q}")
    print(f"Expanded: {expanded}")

Query expansion results:

Original: What is Apple's total revenue for 2023?
Expanded: What is Apple's total revenue for 2023? revenue net sales total sales income total revenue net sales total net sales

Original: What is Apple's net income?
Expanded: What is Apple's net income? net income profit earnings after tax

Original: Apple cash flow from operations
Expanded: Apple cash flow from operations cash flow operations operating activities


In [63]:
# Test retrieval with expanded queries
print("Retrieval with query expansion:")
for query in test_queries_expanded:
    expanded = expand_financial_query(query)
    results = retreive_with_citations(expanded, bm25, enriched_texts, chunks, collection)
    print(f"\nQuery: '{query}'")
    for r in results[:2]:
        print(f"  [{r['citation']}]")
        print(f"  Score: {r['hybrid_score']} | Preview: {r['text'][:100]}")

Retrieval with query expansion:

Query: 'What is Apple's total revenue for 2023?'
  [Apple Inc. 10-K FY2023 — Item 6.    [Reserved]]
  Score: 0.736 | Preview: Operating Expenses Operating expenses for 2023, 2022 and 2021 were as follows (dollars in millions):
  [Apple Inc. 10-K FY2023 — Note 2 –Revenue]
  Score: 0.6896 | Preview: For the sale of third-party products where the Company obtains control of the product before transfe

Query: 'What is Apple's net income?'
  [Apple Inc. 10-K FY2023 — Item 6.    [Reserved]]
  Score: 0.6727 | Preview: Operating Expenses Operating expenses for 2023, 2022 and 2021 were as follows (dollars in millions):
  [Apple Inc. 10-K FY2023 — Notes to Consolidated Financial Statements]
  Score: 0.6217 | Preview: Revenue The Company records revenue net of taxes collected from customers that are remitted to gover

Query: 'Apple cash flow from operations'
  [Apple Inc. 10-K FY2023 — CONSOLIDATED STATEMENTS OF CASH FLOWS]
  Score: 0.5395 | Preview: (In millions) 

# Citation Formatter 

In [68]:
def citation_formatted_answer(query: str,
                             retrieved_results: list[dict]) -> str:
    context_parts = []
    
    for i, result in enumerate(retrieved_results):
        context_parts.append(
            f"[Source {i+1}: {result['citation']}]\n{result['text'][:500]}"
        )
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are a financial analyst assistant. Answer the question using ONLY the provided sources below.
For every fact you state, cite the source number in brackets like [Source 1].
If the sources don't contain enough information, say so explicitly.

Sources:
{context}

Question: {query}

Answer:"""
    
    return prompt

In [69]:
query = "What is Apple's cash flow from operations?"
expanded_query = expand_financial_query(query)
results = retreive_with_citations(expanded_query, bm25, enriched_texts, chunks, collection)
prompt = citation_formatted_answer(query, results)
print("Citation-aware prompt:")
print("-" * 50)
print(prompt[:800])
print("...")
print(f"\nTotal prompt length: {len(prompt)} chars")

Citation-aware prompt:
--------------------------------------------------
You are a financial analyst assistant. Answer the question using ONLY the provided sources below.
For every fact you state, cite the source number in brackets like [Source 1].
If the sources don't contain enough information, say so explicitly.

Sources:
[Source 1: Apple Inc. 10-K FY2023 — Note 4 –Financial Instruments]
To protect the Company’s foreign currency–denominated term debt or marketable securities from fluctuations in foreign exchange rates, the Company may use forwards, cross-currency swaps or other instruments. The Company designates these instruments as either cash flow or fair value hedges. As of September 30, 2023, the maximum length of time over which the Company is hedging its exposure to the variability in future cash flows for term debt–related foreign currency transactio
...

Total prompt length: 2030 chars


## Day 63  Better Chunking + Citation Layer

### What I built
- Semantic section extraction using font-weight:700 detection
- XBRL metadata filter (URL detection)
- Citation metadata: company, doc_type, fiscal_year, section_title
- Section title enrichment before embedding
- Query expansion for financial vocabulary
- Citation-aware prompt formatter

### Real problems discovered
1. XBRL format: Apple 10-K uses inline XBRL — no standard HTML tags
2. Tables don't embed well: numbers without prose context score poorly
3. Query-document mismatch: "revenue" query misses income statement table
4. Partial fixes: URL filter, title enrichment, query expansion

### What works
- Cash flow query → CONSOLIDATED STATEMENTS OF CASH FLOWS ✅
- Citation format: "Apple Inc. 10-K FY2023 — Section Title" ✅
- Prompt structure ready for LLM generation ✅

### Root cause of retrieval failures
Financial tables contain numbers, not natural language.
Embedding models trained on text don't represent numbers well.
Fix: convert tables to natural language summaries before embedding.
This is Day 64's work.

### Production insight
Query expansion: expand for retrieval, use original for generation.
Retrieval and generation have different vocabulary needs.